In [52]:
import os
import re
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from pydub import AudioSegment
from scipy.fft import fft, fftfreq
import torch.nn.functional as F


# ---------------------------------
# 1. CNN Model Definition (Unchanged)
# ---------------------------------
class CNNModel(nn.Module):
    def __init__(self, in_channels=1):
        super(CNNModel, self).__init__()
        
        # Block 1
        self.conv1 = nn.Conv2d(in_channels=1, 
                               out_channels=1, 
                               kernel_size=(1, 2), 
                               stride=(1, 1), 
                               padding=(0, 0))
        self.relu1 = nn.ReLU()
        
        self.conv2 = nn.Conv2d(in_channels=1, 
                               out_channels=2, 
                               kernel_size=(1, 2), 
                               stride=(1, 1), 
                               padding=(0, 0))
        self.relu2 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))
        
        # Block 2
        self.conv3 = nn.Conv2d(in_channels=2, 
                               out_channels=2, 
                               kernel_size=(1, 2), 
                               stride=(1, 1), 
                               padding=(0, 0))
        self.relu3 = nn.ReLU()
        self.conv4 = nn.Conv2d(in_channels=2, 
                               out_channels=4, 
                               kernel_size=(1, 2), 
                               stride=(1, 1), 
                               padding=(0, 0))
        self.relu4 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))
        
        # Block 3
        self.conv5 = nn.Conv2d(in_channels=4, 
                               out_channels=4, 
                               kernel_size=(1, 2), 
                               stride=(1, 1), 
                               padding=(0, 0))
        self.relu5 = nn.ReLU()
        self.conv6 = nn.Conv2d(in_channels=4, 
                               out_channels=8, 
                               kernel_size=(1, 2), 
                               stride=(1, 1), 
                               padding=(0, 0))
        self.relu6 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))

        # Block 4
        self.conv7 = nn.Conv2d(in_channels=8, 
                               out_channels=8, 
                               kernel_size=(1, 2), 
                               stride=(1, 1), 
                               padding=(0, 0))
        self.relu7 = nn.ReLU()
        self.conv8 = nn.Conv2d(in_channels=8, 
                               out_channels=16, 
                               kernel_size=(1, 2), 
                               stride=(1, 1), 
                               padding=(0, 0))
        self.relu8 = nn.ReLU()
        self.pool4 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))

        # Block 5
        self.conv9 = nn.Conv2d(in_channels=16, 
                               out_channels=16, 
                               kernel_size=(1, 2), 
                               stride=(1, 1), 
                               padding=(0, 0))
        self.relu9 = nn.ReLU()
        self.conv10 = nn.Conv2d(in_channels=16, 
                                out_channels=16, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu10 = nn.ReLU()
        self.pool5 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))

        # Block 6
        self.conv11 = nn.Conv2d(in_channels=16, 
                                out_channels=16, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu11 = nn.ReLU()
        self.conv12 = nn.Conv2d(in_channels=16, 
                                out_channels=32, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu12 = nn.ReLU()
        self.pool6 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))

        # Block 7
        self.conv13 = nn.Conv2d(in_channels=32, 
                                out_channels=32, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu13 = nn.ReLU()
        self.conv14 = nn.Conv2d(in_channels=32, 
                                out_channels=32, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu14 = nn.ReLU()
        self.pool7 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))

        # Block 8
        self.conv15 = nn.Conv2d(in_channels=32, 
                                out_channels=32, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu15 = nn.ReLU()
        self.conv16 = nn.Conv2d(in_channels=32, 
                                out_channels=64, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu16 = nn.ReLU()
        self.pool8 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))

        # Block 9
        self.conv17 = nn.Conv2d(in_channels=64, 
                                out_channels=64, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu17 = nn.ReLU()
        self.conv18 = nn.Conv2d(in_channels=64, 
                                out_channels=64, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu18 = nn.ReLU()
        self.conv19 = nn.Conv2d(in_channels=64, 
                                out_channels=128, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu19 = nn.ReLU()
        self.pool9 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))

        # Block 10
        self.conv20 = nn.Conv2d(in_channels=128, 
                                out_channels=128, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu20 = nn.ReLU()
        self.conv21 = nn.Conv2d(in_channels=128, 
                                out_channels=128, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu21 = nn.ReLU()
        self.conv22 = nn.Conv2d(in_channels=128, 
                                out_channels=128, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu22 = nn.ReLU()
        self.conv23 = nn.Conv2d(in_channels=128, 
                                out_channels=256, 
                                kernel_size=(1, 2), 
                                stride=(1, 1), 
                                padding=(0, 0))
        self.relu23 = nn.ReLU()
        self.pool10 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))

        self.flatten = nn.Flatten()
        # Adjust input dimension if needed
        self.fc1 = nn.Linear(12288, 2048)
        self.fc2 = nn.Linear(2048, 256)
        self.fc3 = nn.Linear(256, 8)
        self.s = nn.Softmax(dim=1)

    def forward(self, x):
        # Block 1
        x = self.pool1(self.relu2(self.conv2(self.relu1(self.conv1(x)))))
        # Block 2
        x = self.pool2(self.relu4(self.conv4(self.relu3(self.conv3(x)))))
        # Block 3
        x = self.pool3(self.relu6(self.conv6(self.relu5(self.conv5(x)))))
        # Block 4
        x = self.pool4(self.relu8(self.conv8(self.relu7(self.conv7(x)))))
        # Block 5
        x = self.pool5(self.relu10(self.conv10(self.relu9(self.conv9(x)))))
        # Block 6
        x = self.pool6(self.relu12(self.conv12(self.relu11(self.conv11(x)))))
        # Block 7
        x = self.pool7(self.relu14(self.conv14(self.relu13(self.conv13(x)))))
        # Block 8
        x = self.pool8(self.relu16(self.conv16(self.relu15(self.conv15(x)))))
        # Block 9
        x = self.pool9(self.relu19(self.conv19(self.relu18(self.conv18(self.relu17(self.conv17(x)))))))
        # Block 10
        x = self.pool10(self.relu23(self.conv23(self.relu22(
            self.conv22(self.relu21(
                self.conv21(self.relu20(
                    self.conv20(x)
                ))
            ))
        ))))

        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x)
        torch.save(model.state_dict(), "model_weights.pt")
        return x

In [53]:
def transform_raw_wav(file_path, target_sr=16000):
    audio = AudioSegment.from_wav(file_path)
    audio = audio.set_frame_rate(target_sr)
    audio = audio.set_channels(1)
    audio_data_bytes = np.array(audio.get_array_of_samples(), dtype=np.int16).tobytes()
    samples = np.frombuffer(audio_data_bytes, dtype=np.int16)
    return samples

def fouriertransform(y, sample_rate=16000):
    N = len(y)
    T = 1.0 / sample_rate
    yf = fft(y)
    xf = fftfreq(N, T)
    return [xf[:N//2], 2.0/N * np.abs(yf[:N//2])]

def load_dataset_and_labels(dataset_dir="dataset"):
    """
    Load all .wav files from subfolders 1..8, parse device_i_j, 
    group device_1_j..device_4_j into a (4,16000) matrix, 
    and assign label = folder-1.
    """
    X, Y = [], []
    direction_folders = [str(i) for i in range(1, 9)]

    for direction_str in direction_folders:
        folder_path = os.path.join(dataset_dir, direction_str)
        if not os.path.exists(folder_path):
            continue
        
        grouped = {}
        for fname in os.listdir(folder_path):
            if not fname.endswith(".wav"):
                continue
            
            match = re.match(r"device_(\d)_(\d+)_", fname)
            if match:
                i = int(match.group(1))  # device # (1..4)
                j = match.group(2)      # set index
                full_path = os.path.join(folder_path, fname)
                
                samples = transform_raw_wav(full_path, target_sr=16000)
                _, amp = fouriertransform(samples, sample_rate=16000)
                amp_16k = amp[:16000]
                
                if j not in grouped:
                    grouped[j] = {}
                grouped[j][i] = amp_16k
        
        # Only keep sets with devices 1..4
        for j_key, device_dict in grouped.items():
            if all(k in device_dict for k in [1, 2, 3, 4]):
                mat = np.stack([
                    device_dict[1],
                    device_dict[2],
                    device_dict[3],
                    device_dict[4],
                ], axis=0)
                X.append(mat)
                label = int(direction_str) - 1  # 0..7
                Y.append(label)
    return X, Y

In [54]:
if __name__ == "__main__":
    # Instantiate the model
    model = CNNModel(in_channels=1)
    
    # Define loss & optimizer
    criterion = nn.CrossEntropyLoss()  # Expects raw logits
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Load dataset
    X, Y = load_dataset_and_labels(dataset_dir="dataset") 
    X_np = np.array(X)  # shape (N, 4, 16000)
    Y_np = np.array(Y)  # shape (N,)

    print(f"Total examples: {X_np.shape[0]}")
    print(f"X_np shape: {X_np.shape}") 
    print(f"Y_np shape: {Y_np.shape}") 

    # Convert to Torch tensors
    X_tensor = torch.tensor(X_np, dtype=torch.float32).unsqueeze(1)  # (N,1,4,16000)
    Y_tensor = torch.tensor(Y_np, dtype=torch.long)                  # (N,)

    # Simple training loop
    epochs = 5
    batch_size = 13
    
    model.train()
    N = len(X_tensor)
    for epoch in range(epochs):
        for start_idx in range(0, N, batch_size):
            end_idx = start_idx + batch_size
            if end_idx > N:
                break
            
            Xbatch = X_tensor[start_idx:end_idx]  # shape (B,1,4,16000)
            ybatch = Y_tensor[start_idx:end_idx]  # shape (B,)

            # Forward pass (returns raw logits now)
            logits = model(Xbatch)  # shape (B,8)

            # Loss
            loss = criterion(logits, ybatch)
            
            # Backprop
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            print(f"Epoch [{epoch+1}/{epochs}], Step [{start_idx+1}/{N}], Loss: {loss.item():.4f}")
    torch.save(model.state_dict(), "model_weights.pt")
    print("Training finished!")

Total examples: 247
X_np shape: (247, 4, 16000)
Y_np shape: (247,)
Epoch [1/5], Step [1/247], Loss: 1.9520
Epoch [1/5], Step [14/247], Loss: 0.0461
Epoch [1/5], Step [27/247], Loss: 9.1888
Epoch [1/5], Step [40/247], Loss: 8.2839
Epoch [1/5], Step [53/247], Loss: 3.7790
Epoch [1/5], Step [66/247], Loss: 4.5607
Epoch [1/5], Step [79/247], Loss: 3.9490
Epoch [1/5], Step [92/247], Loss: 5.0563
Epoch [1/5], Step [105/247], Loss: 4.2852
Epoch [1/5], Step [118/247], Loss: 3.5904
Epoch [1/5], Step [131/247], Loss: 3.4749
Epoch [1/5], Step [144/247], Loss: 3.3495
Epoch [1/5], Step [157/247], Loss: 3.2411
Epoch [1/5], Step [170/247], Loss: 2.8776
Epoch [1/5], Step [183/247], Loss: 2.8456
Epoch [1/5], Step [196/247], Loss: 2.7880
Epoch [1/5], Step [209/247], Loss: 2.7029
Epoch [1/5], Step [222/247], Loss: 2.7995
Epoch [1/5], Step [235/247], Loss: 2.5991
Epoch [2/5], Step [1/247], Loss: 3.0843
Epoch [2/5], Step [14/247], Loss: 3.0691
Epoch [2/5], Step [27/247], Loss: 2.9008
Epoch [2/5], Step [40/

In [55]:
def load_and_transform_4devices(device_files):
    """
    device_files: list or tuple of 4 paths:
      [device_1_file, device_2_file, device_3_file, device_4_file]
    Returns:
      A NumPy array of shape (4, 16000) that can be fed into the CNN.
    """
    assert len(device_files) == 4, "Provide exactly 4 device files in order."
    
    stacked = []
    for fp in device_files:
        samples = transform_raw_wav(fp, target_sr=16000)
        _, amp = fouriertransform(samples, sample_rate=16000)
        amp_16k = amp[:16000]  # keep first 16,000 amplitude values
        stacked.append(amp_16k)
    
    matrix = np.stack(stacked, axis=0)  # shape: (4, 16000)
    return matrix

def predict_single_set(model, device_files):
    """
    Load & transform a single set of 4 .wav files, pass through the model,
    and return the softmax probabilities and predicted class index.
    """
    # (4, 16000)
    mat = load_and_transform_4devices(device_files)

    # Reshape for CNN: (batch_size=1, in_channels=1, 4, 16000)
    mat_tensor = torch.tensor(mat, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    # => shape (1, 1, 4, 16000)

    model.eval()
    with torch.no_grad():
        logits = model(mat_tensor)       # shape (1,8) if you have 8 classes
        probs = torch.softmax(logits, 1) # shape (1,8), sums to 1
        predicted_class = torch.argmax(probs, dim=1).item()

    return probs.squeeze(0).numpy(), predicted_class  # (8,), int

if __name__ == "__main__":
    # 1) Load your trained model
    # Replace 'your_cnn_file' and 'CNNModel' with your actual references
    model = CNNModel(in_channels=1)
    model.load_state_dict(torch.load("model_weights.pt", map_location="cpu"))
    model.eval()

    # 2) Provide paths to your custom .wav files for one set
    # The ordering must be device_1, device_2, device_3, device_4
    device_1_file = "dataset/4/device_1_16_20241106_185835_541427.wav"
    device_2_file = "dataset/4/device_2_16_20241106_185835_540427.wav"
    device_3_file = "dataset/4/device_3_16_20241106_185835_541427.wav"
    device_4_file = "dataset/4/device_4_16_20241106_185835_556564.wav"
    device_files = [device_1_file, device_2_file, device_3_file, device_4_file]

    # 3) Run prediction on this single set
    probabilities, predicted_cls = predict_single_set(model, device_files)

    # 4) Print out results
    print("Probabilities for each of the 8 classes:\n", probabilities)
    print(f"Predicted class index: {predicted_cls}")

/var/folders/cw/6lv3zk_914l3fhljrz4_r0vh0000gn/T/ipykernel_60817/3718593185.py:44: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("model_weig

RuntimeError: Error(s) in loading state_dict for CNNModel:
	Missing key(s) in state_dict: "conv4.weight", "conv4.bias", "conv5.weight", "conv5.bias", "conv6.weight", "conv6.bias", "conv7.weight", "conv7.bias", "conv8.weight", "conv8.bias", "conv9.weight", "conv9.bias", "conv10.weight", "conv10.bias", "conv11.weight", "conv11.bias", "conv12.weight", "conv12.bias", "conv13.weight", "conv13.bias", "conv14.weight", "conv14.bias", "conv15.weight", "conv15.bias", "conv16.weight", "conv16.bias", "conv17.weight", "conv17.bias", "conv18.weight", "conv18.bias", "conv19.weight", "conv19.bias", "conv20.weight", "conv20.bias", "conv21.weight", "conv21.bias", "conv22.weight", "conv22.bias", "conv23.weight", "conv23.bias", "fc3.weight", "fc3.bias". 
	Unexpected key(s) in state_dict: "bn1.weight", "bn1.bias", "bn1.running_mean", "bn1.running_var", "bn1.num_batches_tracked", "bn2.weight", "bn2.bias", "bn2.running_mean", "bn2.running_var", "bn2.num_batches_tracked", "bn3.weight", "bn3.bias", "bn3.runnin